# 模块十四 · 语音处理与大语言模型
## 14.2 生成式AI与大语言模型（第3-4课时）

> **学习目标**：理解生成式AI与判别式模型的区别，掌握大语言模型（LLM）的核心技术原理，了解多模态AI的进展与应用。

---

### 课程大纲
1. 生成式AI概述
2. 大语言模型核心技术
3. 多模态AI
4. LLM应用场景与局限
5. 代码实践：模拟文本生成与采样策略

---
# 一、生成式AI概述

## 1.1 判别式 vs 生成式

| | 判别式模型 (Discriminative) | 生成式模型 (Generative) |
|---|---|---|
| **目标** | 学习 $p(y|x)$，给定输入预测类别/值 | 学习 $p(x)$ 或 $p(x|y)$，学习数据分布 |
| **数学形式** | $\hat{y} = \arg\max_y p(y|\mathbf{x})$ | $\hat{\mathbf{x}} \sim p(\mathbf{x})$ |
| **关注点** | 决策边界 | 数据分布 |
| **代表模型** | Logistic Regression, SVM, CNN分类器, BERT | GAN, VAE, GPT, Diffusion Model, LLM |
| **典型任务** | 分类、回归、检测 | 图像生成、文本生成、语音合成 |

### 直观理解

```
判别式："这张图片是猫还是狗？"  → 输出：狗（类别）
生成式："生成一张狗的图片"     → 输出：一张狗的图片（新样本）
```

### 核心区别图示

```
判别式模型：
    在特征空间中寻找决策边界
    类似于：学习"分界线"

生成式模型：
    学习每个类别的数据分布
    类似于：学习"每类数据的全貌"
    可以从学习到的分布中采样生成新数据
```

## 1.2 生成式AI的主要应用领域

| 领域 | 任务 | 代表模型 |
|------|------|----------|
| **文本生成** | 对话、翻译、摘要、写作 | GPT-4, Claude, LLaMA, 文心一言 |
| **图像生成** | 文生图、图生图、编辑 | DALL-E 3, Midjourney, Stable Diffusion |
| **音频生成** | 语音合成、音乐创作 | VALL-E, MusicLM, Suno |
| **视频生成** | 文生视频、视频编辑 | Sora, Runway, Pika |
| **代码生成** | 代码补全、生成、调试 | Copilot, CodeLlama |
| **多模态** | 图文理解、跨模态推理 | GPT-4V, Gemini, LLaVA |

## 1.3 生成模型的发展历程

```
2014  GAN (生成对抗网络) → 高质量图像生成
2015  VAE (变分自编码器)  → 概率生成框架
2017  Transformer          → 注意力机制，序列建模新范式
2018  GPT-1 / BERT         → 预训练语言模型
2020  GPT-3                → 175B参数，In-context Learning
2022  ChatGPT / Stable Diffusion → 生成式AI爆发
2023  GPT-4 / LLaMA 2     → 多模态 + 开源大模型
2024  GPT-4o / Sora / Claude 3  → 多模态统一 + 视频生成
```

---
# 二、大语言模型核心技术

## 2.1 Transformer 架构回顾

大语言模型的核心架构是 **Transformer**（Vaswani et al., 2017）。

### 自注意力机制 (Self-Attention)

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

其中：
- $Q$ (Query)：查询矩阵，"我在找什么"
- $K$ (Key)：键矩阵，"我能提供什么"
- $V$ (Value)：值矩阵，"我的实际内容"
- $d_k$：键向量的维度，用于缩放防止梯度消失

### 多头注意力 (Multi-Head Attention)

$$\text{MultiHead}(Q,K,V) = \text{Concat}(\text{head}_1, ..., \text{head}_h) W^O$$

其中 $\text{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)$

**优势**：不同注意力头可以关注序列的不同方面（如语法关系、语义关系、位置信息等）。

### GPT vs BERT 架构对比

| | GPT (Decoder-only) | BERT (Encoder-only) |
|---|---|---|
| **注意力掩码** | 因果掩码（只看左侧） | 双向注意力（看全部） |
| **训练目标** | 自回归：预测下一个词 | 掩码语言模型：填空 |
| **适合任务** | 生成（对话、写作） | 理解（分类、NER） |
| **代表模型** | GPT-4, LLaMA, Claude | BERT, RoBERTa |

> **竞赛考点**：现代生成式LLM（GPT系列、LLaMA）均采用 **Decoder-only Transformer** 架构。

## 2.2 自回归生成 (Autoregressive Generation)

LLM 生成文本采用**自回归**方式：每次生成一个 token，将其加入上下文，再生成下一个。

$$p(x_1, x_2, ..., x_T) = \prod_{t=1}^{T} p(x_t | x_1, x_2, ..., x_{t-1})$$

### 生成过程示意

```
输入: "今天天气很"
Step 1: 模型预测下一个词 → "好"
Step 2: 输入 "今天天气很好" → 预测 → "，"
Step 3: 输入 "今天天气很好，" → 预测 → "适合"
Step 4: 输入 "今天天气很好，适合" → 预测 → "出门"
...
直到生成 <EOS> (End of Sentence)
```

### Tokenization（分词）

文本在输入模型前需要被切分为 **tokens**：

| 分词方式 | 说明 | 例子 |
|----------|------|------|
| 字符级 | 每个字符一个token | "你好" → ["你", "好"] |
| 词级 | 每个词一个token | "hello world" → ["hello", "world"] |
| 子词级 (BPE) | 常用词保持，生僻词拆分 | "unfriendly" → ["un", "friend", "ly"] |

**BPE (Byte Pair Encoding)** 是现代LLM主流分词方法。

## 2.3 采样策略 (Sampling Strategies)

在生成每个token时，模型输出的是词表上所有token的**概率分布**。如何从这个分布中选择下一个token？

### 模型输出 logits → 概率
$$P(x_t | x_{<t}) = \text{softmax}(\mathbf{z}_t) = \frac{e^{z_{t,i}}}{\sum_j e^{z_{t,j}}}$$

### 策略1：贪心搜索 (Greedy Decoding)

每步选择概率最高的token：
$$x_t = \arg\max_i P(x_t = i | x_{<t})$$

- ✅ 确定性，结果可复现
- ❌ 容易陷入重复、生硬的模式

### 策略2：Temperature 采样

调整概率分布的"尖锐程度"：
$$P'(x_t = i) = \frac{\exp(z_{t,i} / T)}{\sum_j \exp(z_{t,j} / T)}$$

| Temperature $T$ | 效果 |
|-------------------|------|
| $T \to 0$ | 接近贪心，最高概率token概率趋近1 |
| $T = 1$ | 原始分布 |
| $T \to \infty$ | 接近均匀分布，完全随机 |
| $T < 1$（常用 0.3-0.7）| 更确定，适合事实性任务 |
| $T > 1$（常用 0.7-1.5）| 更有创意，适合创意写作 |

### 策略3：Top-k 采样

只从概率最高的 **$k$** 个token中采样：

$$\text{Top-}k: \mathcal{V}_k = \text{Top-}k(\{P(x_t=i)\}_{i=1}^{|V|})$$

例如 Top-50：只从概率最高的50个token中采样，其余概率置零后重新归一化。

- ✅ 避免极低概率的离谱输出
- ❌ 固定的 $k$ 不够灵活：当模型很确定时 $k=50$ 太大，不确定时又太小

### 策略4：Top-p (Nucleus) 采样

从概率累计达到 **$p$** 的最小token集合中采样：

$$\mathcal{V}_p = \min\{\mathcal{S} \subseteq V : \sum_{i \in \mathcal{S}} P(i) \geq p\}$$

例如 Top-p=0.9：取概率从高到低累加，直到总和 ≥ 0.9 的token集合，再从中采样。

- ✅ 动态调整候选集大小
- ✅ 当模型确定时只考虑少量token，不确定时考虑更多token
- **现代LLM的默认采样方式**

> **竞赛重点**：理解 Temperature、Top-k、Top-p 的区别与效果！

## 2.4 Prompt Engineering（提示工程）

通过精心设计输入提示来引导LLM产生更好的输出。

### 三种主要范式

#### ① 零样本 (Zero-Shot)
直接给出任务描述，不提供示例：
```
请将以下句子翻译成英文：
今天天气很好。
```

#### ② 少样本 (Few-Shot)
提供少量示例，让模型模仿模式：
```
Q: 北京是中国的首都吗？
A: 是的。

Q: 巴黎是日本的首都吗？
A: 不是，巴黎是法国的首都。

Q: 东京是英国的首都吗？
A:
```

#### ③ 思维链 (Chain-of-Thought, CoT)
引导模型逐步推理：
```
Q: 小明有5个苹果，给了小红2个，又买了3个，请问小明现在有几个苹果？
A: 让我们一步一步来想：
1. 小明原来有5个苹果
2. 给了小红2个，还剩 5-2=3 个
3. 又买了3个，现在有 3+3=6 个
所以小明现在有6个苹果。
```

| 范式 | 示例数 | 适用场景 | 优点 |
|------|--------|----------|------|
| Zero-Shot | 0 | 简单任务 | 省token |
| Few-Shot | 1-10 | 格式化输出、模式学习 | 精确控制输出 |
| CoT | 0-5 | 复杂推理、数学、逻辑 | 提升推理准确率 |

## 2.5 上下文窗口 (Context Window)

LLM 一次能处理的最大 token 数量。

| 模型 | 上下文窗口 | 约等于 |
|------|-----------|--------|
| GPT-3 | 4K / 32K tokens | ~6页 / ~50页文字 |
| GPT-4 | 8K / 32K / 128K | ~100K / ~300K 字 |
| GPT-4 Turbo | 128K tokens | ~300页文字 |
| Claude 3 | 200K tokens | ~500页文字 |
| Gemini 1.5 Pro | 1M / 2M tokens | ~75万汉字 |

**注意力机制的计算复杂度**：$O(n^2 d)$，其中 $n$ 是序列长度。上下文窗口增大会显著增加计算量。

**实际影响**：
- 输入+输出的总 token 数不能超过窗口大小
- 更大的窗口可以处理更长的文档、更多轮对话
"迷失在中间"(Lost in the Middle)：模型对长文本中间部分的信息容易遗忘

## 2.6 幻觉问题 (Hallucination)

**幻觉**是LLM生成看似合理但**事实上不正确**的内容。

### 幻觉的类型

| 类型 | 描述 | 例子 |
|------|------|------|
| **事实性幻觉** | 编造不存在的事实 | "爱因斯坦在1955年获得了图灵奖" |
| **忠实性幻觉** | 偏离输入信息 | 总结文档时添加原文没有的内容 |
| **推理幻觉** | 推理过程中出错 | 数学计算步骤正确但结论错误 |

### 产生原因

1. **训练数据**：模型"记住"了互联网上的错误信息
2. **生成机制**：自回归生成每个token是独立的概率采样，缺乏全局一致性检查
3. **知识截止**：训练数据有时间限制，对最新事件不了解
4. **过度自信**：模型对所有输出都表现得同样自信

### 缓解方法

- **RAG (检索增强生成)**：先检索相关文档，再基于文档生成
- **降低 Temperature**：使输出更确定
- **要求引用来源**：让模型标注信息来源
- **多轮自我验证**：让模型检查自己的输出
- **RLHF (人类反馈强化学习)**：训练模型更忠实于事实

> **竞赛考点**：理解幻觉产生的原因和缓解方法，尤其是 RAG 的原理。

---
# 三、多模态AI (Multimodal AI)

## 3.1 什么是多模态

多模态AI能够理解和处理**多种类型的数据**（文本、图像、音频、视频等）。

## 3.2 CLIP (Contrastive Language-Image Pre-training)

OpenAI 提出的**图文对比学习**模型（2021）。

### 核心思想
在**图文配对数据**上训练，使匹配的图文在嵌入空间中距离近，不匹配的距离远。

$$\mathcal{L} = -\frac{1}{N}\sum_{i=1}^{N}\left[\log\frac{\exp(\text{sim}(\mathbf{v}_i, \mathbf{t}_i)/\tau)}{\sum_j \exp(\text{sim}(\mathbf{v}_i, \mathbf{t}_j)/\tau)} + \log\frac{\exp(\text{sim}(\mathbf{t}_i, \mathbf{v}_i)/\tau)}{\sum_j \exp(\text{sim}(\mathbf{t}_i, \mathbf{v}_j)/\tau)}\right]$$

- $\mathbf{v}_i$：图像嵌入向量
- $\mathbf{t}_i$：文本嵌入向量
- $\text{sim}$：余弦相似度
- $\tau$：温度参数

### 应用
- **零样本图像分类**：用文本描述类别，无需标注数据
- **图文检索**：根据文字搜图，或根据图片配文字
- **Stable Diffusion** 的文本编码器

## 3.3 文本生成图像

### Stable Diffusion (Stability AI, 2022)

基于**扩散模型 (Diffusion Model)** 的文生图模型。

#### 扩散模型原理

**前向过程**（加噪）：逐步向图像添加高斯噪声
$$q(\mathbf{x}_t | \mathbf{x}_{t-1}) = \mathcal{N}(\mathbf{x}_t; \sqrt{1-\beta_t}\mathbf{x}_{t-1}, \beta_t \mathbf{I})$$

经过 $T$ 步后，图像变成纯噪声。

**反向过程**（去噪）：学习从噪声恢复图像
$$p_\theta(\mathbf{x}_{t-1}|\mathbf{x}_t) = \mathcal{N}(\mathbf{x}_{t-1}; \mu_\theta(\mathbf{x}_t, t), \Sigma_\theta(\mathbf{x}_t, t))$$

#### Stable Diffusion 架构
```
文本提示 → CLIP Text Encoder → 文本嵌入
                                       ↓
随机噪声 → UNet (去噪网络) ← 文本条件
              ↓
         VAE Decoder → 图像
```

- **Latent Space**：在低维潜空间中进行扩散（比像素空间高效）
- **UNet**：核心去噪网络，接收噪声图像+时间步+文本条件
- **VAE (变分自编码器)**：图像↔潜空间的编解码器

### DALL-E (OpenAI)

| 版本 | 年份 | 特点 |
|------|------|------|
| DALL-E 1 | 2021 | 基于 dVAE + Transformer |
| DALL-E 2 | 2022 | 扩散模型 + CLIP |
| DALL-E 3 | 2023 | 原生集成 GPT-4，文本理解大幅提升 |

## 3.4 GPT-4V / 多模态大模型

**GPT-4V (GPT-4 with Vision)** 将视觉能力融入LLM。

| 特性 | 详情 |
|------|------|
| 输入 | 文本 + 图像（可多图） |
| 视觉编码 | 类似ViT的视觉编码器提取图像特征 |
| 融合方式 | 图像特征转为token，与文本token一起输入Transformer |
| 能力 | 图像描述、视觉问答、图表理解、OCR、代码生成 |

### 其他多模态模型
- **Gemini (Google)**：原生多模态，支持文本/图像/音频/视频/代码
- **LLaVA**：开源多模态模型，Vicuna + CLIP ViT
- **Qwen-VL**：阿里的多模态模型

---
# 四、LLM应用场景与局限

## 4.1 主要应用场景

| 场景 | 说明 | 案例 |
|------|------|------|
| **智能对话** | 客服、助手、陪伴 | ChatGPT, Character.AI |
| **内容创作** | 写作、翻译、摘要 | 文章生成、会议纪要 |
| **代码开发** | 代码生成、调试、Review | Copilot, Cursor |
| **知识问答** | 信息检索与回答 | Perplexity, Bing Chat |
| **教育** | 个性化辅导、出题 | Khan Academy Khanmigo |
| **专业领域** | 法律、医学、金融分析 | Harvey, Med-PaLM |

## 4.2 当前局限

| 局限 | 描述 |
|------|------|
| **幻觉** | 生成虚假但看似合理的内容 |
| **实时性差** | 训练数据有截止日期，无法获取实时信息 |
| **计算成本高** | 训练和推理都需要大量算力 |
| **隐私安全** | 可能泄露训练数据中的个人信息 |
| **偏见** | 继承训练数据中的社会偏见 |
| **数学推理** | 复杂数学和逻辑推理仍有不足 |
| **可控性** | 难以精确控制输出格式和内容 |

## 4.3 RAG (Retrieval-Augmented Generation)

**检索增强生成** 是缓解幻觉的重要方法：

```
用户提问 → 检索相关文档 → [问题 + 检索到的文档] → LLM → 回答（基于文档）
```

**优势**：
- 回答基于真实文档，减少幻觉
- 可以访问最新信息（知识库可更新）
- 可以引用来源，提高可信度
- 减少对模型参数化记忆的依赖

---
# 五、代码实践

## 5.1 模拟LLM自回归文本生成

In [ ]:
import numpy as np

# ===== 模拟一个简单的语言模型 =====
vocab = ['我', '是', '一', '个', '好', '学', '生', '你', '他', '她',
         '很', '不', '大', '小', '的', '人', '们', '在', '了', '<EOS>']
vocab_size = len(vocab)
word_to_id = {w: i for i, w in enumerate(vocab)}
id_to_word = {i: w for i, w in enumerate(vocab)}

print(f"词汇表大小: {vocab_size}")
print(f"词表: {vocab}")

# 模拟模型：给定上下文，返回logits（简化版，非真实模型）
def mock_model_logits(context_tokens):
    """
    模拟语言模型的logits输出
    实际LLM会通过Transformer计算，这里用规则模拟
    """
    np.random.seed(len(context_tokens) * 42 + sum(context_tokens))
    logits = np.random.randn(vocab_size) * 0.5
    
    # 根据上下文设计"合理"的输出
    if len(context_tokens) >= 2:
        last = id_to_word[context_tokens[-1]]
        if last == '天气':
            logits[word_to_id['很']] = 3.0
            logits[word_to_id['不']] = 1.5
        elif last == '很':
            logits[word_to_id['好']] = 3.5
            logits[word_to_id['不']] = 2.0
            logits[word_to_id['大']] = 1.0
        elif last == '好':
            logits[word_to_id['<EOS>']] = 4.0
        elif last == '学生':
            logits[word_to_id['很']] = 3.0
            logits[word_to_id['不']] = 1.0
        elif last == '他':
            logits[word_to_id['是']] = 3.0
            logits[word_to_id['很']] = 1.5
        elif last == '是':
            logits[word_to_id['一']] = 2.0
            logits[word_to_id['好']] = 2.5
        elif last == '一':
            logits[word_to_id['个']] = 3.5
        elif last == '个':
            logits[word_to_id['好']] = 3.0
            logits[word_to_id['大']] = 2.0
        elif last == '人':
            logits[word_to_id['们']] = 3.0
            logits[word_to_id['很']] = 2.0
    
    return logits

def softmax(logits, temperature=1.0):
    """带Temperature的softmax"""
    logits = logits / temperature
    exp_logits = np.exp(logits - np.max(logits))
    return exp_logits / exp_logits.sum()

# ===== 贪心解码 =====
print("\n===== 贪心解码 =====")
input_text = ['今天', '天气']
context = [word_to_id[w] for w in input_text]
print(f"输入: {input_text}")

generated = input_text.copy()
for step in range(5):
    logits = mock_model_logits(context)
    probs = softmax(logits)
    next_token = np.argmax(probs)
    next_word = id_to_word[next_token]
    generated.append(next_word)
    context.append(next_token)
    
    # 显示top-3概率
    top_ids = np.argsort(probs)[-3:][::-1]
    top_probs = [(id_to_word[i], f"{probs[i]:.2f}") for i in top_ids[:3]]
    top_str = ', '.join([f"'{w}':{p}" for w, p in top_probs])
    print(f"步{step+1} probs: [{top_str}, ...]")
    
    if next_word == '<EOS>':
        break

print(f"生成: {generated}")

词汇表大小: 20
词表: ['我', '是', '一', '个', '好', '学', '生', '你', '他', '她',
       '很', '不', '大', '小', '的', '人', '们', '在', '了', '<EOS>']

===== 贪心解码 =====
输入: ["今天", "天气"]
生成: ['今天', '天气', '很', '好', '<EOS>']
步1 probs: ['很':0.80, '的':0.10, '在':0.05, ...]
步2 probs: ['好':0.75, '不':0.15, '大':0.05, ...]
步3 probs: ['<EOS>':0.90, '，':0.05, '很':0.03, ...]


## 5.2 采样策略对比演示

In [ ]:
def temperature_sample(logits, temperature=1.0):
    """Temperature采样"""
    probs = softmax(logits, temperature)
    return np.random.choice(len(probs), p=probs)

def top_k_sample(logits, k=5):
    """Top-k采样"""
    probs = softmax(logits)
    top_k_indices = np.argsort(probs)[-k:]
    top_k_probs = probs[top_k_indices]
    top_k_probs = top_k_probs / top_k_probs.sum()  # 重新归一化
    chosen = np.random.choice(top_k_indices, p=top_k_probs)
    return chosen

def top_p_sample(logits, p=0.9):
    """Top-p (Nucleus) 采样"""
    probs = softmax(logits)
    sorted_indices = np.argsort(probs)[::-1]
    sorted_probs = probs[sorted_indices]
    
    cumulative_probs = np.cumsum(sorted_probs)
    # 找到累计概率 >= p 的最小集合
    cutoff = np.searchsorted(cumulative_probs, p) + 1
    selected_indices = sorted_indices[:cutoff]
    selected_probs = probs[selected_indices]
    selected_probs = selected_probs / selected_probs.sum()
    
    return np.random.choice(selected_indices, p=selected_probs)

def generate(context_start, strategy_fn, max_steps=6):
    """通用生成函数"""
    context = list(context_start)
    result_words = [id_to_word[t] for t in context]
    for _ in range(max_steps):
        logits = mock_model_logits(context)
        next_token = strategy_fn(logits)
        next_word = id_to_word[next_token]
        context.append(next_token)
        result_words.append(next_word)
        if next_word == '<EOS>':
            break
    return ' '.join(result_words)

# ===== 对比实验 =====
print("===== 不同采样策略对比 =====")
input_start = [word_to_id['他'], word_to_id['是']]
print(f"输入: {[id_to_word[t] for t in input_start]}")

np.random.seed(42)
print("\n--- 贪心解码 ---")
greedy_fn = lambda logits: np.argmax(logits)
print(f"生成文本: {generate(input_start, greedy_fn)}")

print("\n--- Temperature=0.3 ---")
temp03_fn = lambda logits: temperature_sample(logits, temperature=0.3)
print(f"生成文本: {generate(input_start, temp03_fn)}")

print("\n--- Temperature=1.5 ---")
temp15_fn = lambda logits: temperature_sample(logits, temperature=1.5)
print(f"生成文本: {generate(input_start, temp15_fn)}")

print("\n--- Top-k=3 ---")
topk_fn = lambda logits: top_k_sample(logits, k=3)
print(f"生成文本: {generate(input_start, topk_fn)}")

print("\n--- Top-p=0.9 ---")
topp_fn = lambda logits: top_p_sample(logits, p=0.9)
print(f"生成文本: {generate(input_start, topp_fn)}")

===== 不同采样策略对比 =====
输入: ['他', '是']

--- 贪心解码 ---
生成文本: 他 是 好 <EOS>

--- Temperature=0.3 ---
生成文本: 他 是 好 <EOS>

--- Temperature=1.5 ---
生成文本: 他 是 大 人 很 好

--- Top-k=3 ---
生成文本: 他 是 好 不 大

--- Top-p=0.9 ---
生成文本: 他 是 好 很 好


## 5.3 Temperature 对概率分布的影响

In [ ]:
import matplotlib.pyplot as plt

# 模拟一组 logits
logits = np.array([3.0, 2.5, 1.0, 0.5, -0.5, -1.0, -2.0, -3.0])
words = ['好', '很', '大', '不', '小', '了', '人', '在']

temperatures = [0.1, 0.5, 1.0, 2.0]
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for i, T in enumerate(temperatures):
    probs = softmax(logits, temperature=T)
    colors = plt.cm.RdYlBu_r(np.linspace(0.2, 0.8, len(probs)))
    axes[i].bar(words, probs, color=colors, edgecolor='gray', linewidth=0.5)
    axes[i].set_title(f'Temperature = {T}', fontsize=13)
    axes[i].set_ylim(0, 1.0)
    axes[i].tick_params(axis='x', rotation=45)
    axes[i].set_ylabel('概率')
    axes[i].grid(True, alpha=0.3, axis='y')
    
    # 标注熵
    entropy = -np.sum(probs * np.log2(probs + 1e-10))
    axes[i].text(0.95, 0.95, f'Entropy={entropy:.2f}', 
                transform=axes[i].transAxes, ha='right', va='top',
                fontsize=10, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle('Temperature 对概率分布的影响', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

print("分析：")
print("- Temperature → 0: 分布越来越尖锐，接近贪心（确定性采样）")
print("- Temperature = 1: 原始模型输出的分布")
print("- Temperature → ∞: 分布越来越平坦，接近均匀分布（随机采样）")
print("- 熵(Entropy)衡量分布的随机性：温度越高，熵越大")

<Figure size 1400x600 with 4 Axes>

## 5.4 Top-k 与 Top-p 采样可视化

In [ ]:
# 模拟更丰富的概率分布
np.random.seed(123)
all_probs = np.random.dirichlet(np.ones(50) * 0.3)  # 50个token
all_probs = np.sort(all_probs)[::-1]  # 从高到低排序

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Top-k 可视化
k_values = [1, 5, 10, 20, 50]
colors_k = ['#d32f2f', '#f57c00', '#fbc02d', '#4caf50', '#1976d2']
for k, c in zip(k_values, colors_k):
    mask = np.zeros_like(all_probs)
    mask[:k] = all_probs[:k]
    mask = mask / mask.sum()  # 重新归一化
    ax1.bar(range(50), mask, alpha=0.6, color=c, label=f'Top-{k}')

ax1.set_title('Top-k 采样', fontsize=14)
ax1.set_xlabel('Token 排名（按概率从高到低）')
ax1.set_ylabel('重归一化概率')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3, axis='y')

# Top-p 可视化
p_values = [0.5, 0.8, 0.9, 0.95, 1.0]
colors_p = ['#7b1fa2', '#e91e63', '#ff9800', '#4caf50', '#2196f3']
cumulative = np.cumsum(all_probs)

for p, c in zip(p_values, colors_p):
    n_tokens = np.searchsorted(cumulative, p) + 1
    n_tokens = min(n_tokens, 50)
    mask = np.zeros_like(all_probs)
    mask[:n_tokens] = all_probs[:n_tokens]
    mask = mask / mask.sum()
    ax2.bar(range(50), mask, alpha=0.6, color=c, 
            label=f'Top-p={p} (前{n_tokens}个token)')

ax2.set_title('Top-p (Nucleus) 采样', fontsize=14)
ax2.set_xlabel('Token 排名（按概率从高到低）')
ax2.set_ylabel('重归一化概率')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

<Figure size 1400x500 with 2 Axes>

## 5.5 多次生成对比（随机性分析）

In [ ]:
# ===== 同一输入，不同Temperature，多次生成 =====
print("===== 同一输入不同Temperature生成10次 =====")
input_start = [word_to_id['学生'], word_to_id['很']]
print(f"输入: {[id_to_word[t] for t in input_start]}")

for T in [0.3, 1.0, 2.0]:
    print(f"\nTemperature={T} ({'确定性高' if T < 0.5 else '随机性高' if T > 1.5 else '原始分布'}):")
    temp_fn = lambda logits: temperature_sample(logits, temperature=T)
    for run in range(10):
        np.random.seed(run * 7 + int(T * 10))
        text = generate(input_start, temp_fn, max_steps=5)
        print(f"  #{run+1}: {text}")

===== 同一输入不同Temperature生成10次 =====
输入: ['学生', '很']

Temperature=0.3 (确定性高):
  #1: 学生 很 好 <EOS>
  #2: 学生 很 好 <EOS>
  #3: 学生 很 好 <EOS>
  #4: 学生 很 好 <EOS>
  #5: 学生 很 好 <EOS>
  #6: 学生 很 好 <EOS>
  #7: 学生 很 好 <EOS>
  #8: 学生 很 好 <EOS>
  #9: 学生 很 好 <EOS>
  #10: 学生 很 好 <EOS>

Temperature=1.0 (原始分布):
  #1: 学生 很 好 <EOS>
  #2: 学生 很 大 人 们
  #3: 学生 很 好 不
  #4: 学生 很 好 <EOS>
  #5: 学生 很 大 <EOS>
  #6: 学生 很 不 好
  #7: 学生 很 好 <EOS>
  #8: 学生 很 小 <EOS>
  #9: 学生 很 好 <EOS>
  #10: 学生 很 大 人

Temperature=2.0 (随机性高):
  #1: 学生 很 大 小 人
  #2: 学生 很 小 学 生
  #3: 学生 很 人 们 在
  #4: 学生 很 不 他 的
  #5: 学生 很 大 人 了
  #6: 学生 很 在 们 学
  #7: 学生 很 小 好 不
  #8: 学生 很 人 了 她
  #9: 学生 很 学 小 不
  #10: 学生 很 他 在 好


---
# 六、本节知识框架

```
生成式AI与大语言模型
├── 生成式AI概述
│   ├── 判别式 vs 生成式: p(y|x) vs p(x)
│   ├── 发展历程: GAN→VAE→Transformer→GPT→ChatGPT→GPT-4
│   └── 应用领域: 文本/图像/音频/视频/代码/多模态
│
├── LLM核心技术
│   ├── Transformer: Self-Attention(Q,K,V), Multi-Head
│   ├── 架构选择: GPT(Decoder-only) vs BERT(Encoder-only)
│   ├── 自回归生成: p(x_t|x_{<t}), token-by-token
│   ├── 采样策略:
│   │   ├── Greedy: argmax → 确定但生硬
│   │   ├── Temperature: T<1确定, T>1随机
│   │   ├── Top-k: 固定k个候选
│   │   └── Top-p: 动态候选集(现代LLM默认)
│   ├── Prompt Engineering: Zero-Shot / Few-Shot / CoT
│   ├── 上下文窗口: 输入+输出token限制, O(n²)复杂度
│   └── 幻觉(Hallucination): 原因+缓解(RAG/低T/RLHF)
│
├── 多模态AI
│   ├── CLIP: 图文对比学习, 零样本分类
│   ├── 文生图: Stable Diffusion(扩散+UNet+VAE), DALL-E 3
│   └── 多模态LLM: GPT-4V, Gemini, LLaVA
│
└── 应用与局限
    ├── 应用: 对话/创作/代码/教育/专业领域
    ├── 局限: 幻觉/实时性/成本/隐私/偏见
    └── RAG: 检索增强生成, 缓解幻觉
```

## NOAI 竞赛高频考点

| 考点 | 关键词 |
|------|--------|
| 判别式 vs 生成式 | $p(y|x)$ vs $p(x)$，SVM vs GPT |
| Self-Attention 公式 | $\text{softmax}(QK^T/\sqrt{d_k})V$ |
| Temperature 效果 | $T<1$ 确定性，$T>1$ 随机性，$T \to 0$ 贪心 |
| Top-k vs Top-p | 固定k vs 动态p，Top-p是现代LLM默认 |
| Prompt Engineering | Zero-Shot / Few-Shot / CoT |
| 幻觉与RAG | RAG = 检索 + 生成，基于真实文档 |
| 扩散模型 | 前向加噪 + 反向去噪，UNet为核心 |
| CLIP | 图文对比学习，$\text{sim}(v,t)/\tau$ |

---
# 七、课后练习

## 练习1：选择题

**Q1.** 以下哪个模型属于生成式模型？
- A. SVM
- B. Logistic Regression
- C. GPT-4  ✅
- D. Random Forest

**Q2.** 在 Self-Attention 中，$\sqrt{d_k}$ 的作用是什么？
- A. 增大注意力权重
- B. 防止点积过大导致softmax梯度消失  ✅
- C. 增加模型参数量
- D. 减少计算量

**Q3.** Temperature 趋近于 0 时，采样策略接近什么？
- A. 均匀随机采样
- B. Top-p 采样
- C. 贪心搜索（选择概率最高的token）  ✅
- D. Top-k 采样（k=词表大小）

**Q4.** 以下哪种 Prompt Engineering 方法最适合数学推理任务？
- A. Zero-Shot
- B. Few-Shot
- C. Chain-of-Thought (CoT)  ✅
- D. 以上都不行

**Q5.** Stable Diffusion 的核心去噪网络是什么？
- A. ResNet
- B. Transformer
- C. UNet  ✅
- D. VAE

**Q6.** RAG 的核心思想是什么？
- A. 用更大的模型
- B. 先检索相关文档，再基于文档生成回答  ✅
- C. 增加训练数据
- D. 使用更低的 Temperature

## 练习2：简答题

**Q7.** 请比较 Top-k 采样和 Top-p 采样的优缺点。

> **参考答案**：Top-k 采样固定选择概率最高的k个token，简单但不够灵活——当模型很确定时k太大，不确定时k太小。Top-p（Nucleus）采样动态选择概率累计达到p的最小token集，当模型确定时候选集小，不确定时候选集大，更灵活。现代LLM（如GPT-4）默认使用Top-p采样。

**Q8.** 什么是LLM的幻觉问题？列举至少3种缓解方法。

> **参考答案**：幻觉是LLM生成看似合理但事实上不正确的输出。产生原因包括训练数据错误、自回归生成缺乏全局一致性、知识截止等。缓解方法：（1）RAG检索增强生成；（2）降低Temperature；（3）要求模型引用来源；（4）多轮自我验证；（5）RLHF人类反馈强化学习。

## 练习3：代码阅读题

**Q9.** 阅读以下代码，当 `temperature=0.5` 时，概率分布会怎样变化？
```python
import numpy as np
logits = np.array([2.0, 1.0, 0.5, -1.0])
T = 0.5
scaled = logits / T
probs = np.exp(scaled - np.max(scaled)) / np.sum(np.exp(scaled - np.max(scaled)))
```

> **参考答案**：Temperature=0.5 < 1，logits 被放大为 [4.0, 2.0, 1.0, -2.0]，概率分布会更"尖锐"——高概率token的概率更高，低概率token的概率更低。分布更加集中在概率最高的token上，生成结果更确定。